# Пересчёт слепого переноса LSTM на замороженных входах

Notebook повторяет **только инференс** четырёх ранее обученных LSTM (ERA5, ERA5-Land, GPCP, MSWEP) для 409 постов слепого переноса за 2017–2018 гг. Переобучение не выполняется.

Контракт:

- используются исходные checkpoints и лучшие эпохи 12 / 11 / 13 / 13;
- метеорологические входы берутся из прежнего снимка `data/ws_related_meteo/nc_concat`;
- временная координата `index` нормализуется в `date` в отдельном versioned data-root;
- исходные модели, прогнозы и `test_results.p` не перезаписываются;
- рассчитываются метрики только для 395 постов, не участвовавших в обучении;
- входы, checkpoints, прогнозы и таблицы фиксируются SHA-256.

Запускать на GPU-компьютере из корня репозитория или каталога `notebooks`. Требуются CUDA-версия PyTorch и `neuralhydrology==1.12.0`. Выполнить все ячейки сверху вниз.

In [1]:
from datetime import datetime, timezone
import gc
import hashlib
import importlib.metadata
import json
from pathlib import Path
import platform
import random
import shutil
import sys

import numpy as np
import pandas as pd
import xarray as xr


def find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError("Не найден корень репозитория Dissertation.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA = ROOT / "data"
RUN_TAG = "2026-09-04_frozen_pre_update_v1"
RUN_ROOT = DATA / "res_20260904" / "chapter_three" / "blind_transfer_recalc" / RUN_TAG
FROZEN_DATA = RUN_ROOT / "neuralhydrology_data"
FROZEN_TS = FROZEN_DATA / "time_series"
FROZEN_ATTRIBUTES = FROZEN_DATA / "attributes"
CONTRACTS_DIR = RUN_ROOT / "contracts"
PREDICTIONS_DIR = RUN_ROOT / "predictions"
TABLES_DIR = RUN_ROOT / "tables"

OLD_TS = DATA / "ws_related_meteo" / "nc_concat"
ATTRIBUTES_SOURCE = DATA / "attributes" / "hydro_atlas_cis_camels.csv"
BLIND_SOURCE = DATA / "models" / "blind" / "blind_gauges.txt"
INVENTORY_SOURCE = (
    DATA
    / "res_20260828"
    / "chapter_two"
    / "blind_transfer"
    / "tables"
    / "blind_inventory_audit.csv"
)
PREVIOUS_METRICS = (
    DATA
    / "res_20260828"
    / "chapter_two"
    / "blind_transfer"
    / "tables"
    / "honest_blind_lstm_metrics.csv"
)
OBSERVATIONS_DIR = DATA / "nc_all_q"
INPUT_MANIFEST = FROZEN_DATA / "manifest.json"
COMPLETED_MANIFEST = RUN_ROOT / "COMPLETED.json"

EVALUATION_DATES = pd.date_range("2017-01-01", "2018-12-31", freq="D")
INPUT_DATES = pd.date_range("2008-01-01", "2018-12-31", freq="D")
REQUIRED_VARIABLES = [
    "q_mm_day",
    "prcp_e5",
    "prcp_e5l",
    "prcp_gpcp",
    "prcp_mswep",
    "t_max_e5l",
    "t_min_e5l",
]

MODEL_SPECS = {
    "e5": {
        "label": "ERA5",
        "run_dir": DATA
        / "lstm_configs"
        / "model_runs"
        / "FULL_cudalstm_q_mm_day_256_365_e5_1201_233905",
        "epoch": 12,
        "dynamic_inputs": ["prcp_e5", "t_max_e5l", "t_min_e5l"],
        "config_sha256": "24774d42e76185240e2421accdeb6d77c6bde1711adbdbd6f5ea1a989c334ff4",
        "checkpoint_sha256": "7b930d8f8ef010ef3a13289101dc25a6bcf08f971bed6d209cac91e40e56d2ca",
        "scaler_sha256": "547a362f2629c14178426def3ec80a61edc07ba82b5bb7f43004b9b0f3155731",
    },
    "e5l": {
        "label": "ERA5-Land",
        "run_dir": DATA
        / "lstm_configs"
        / "model_runs"
        / "FULL_cudalstm_q_mm_day_256_365_e5l_1201_193434",
        "epoch": 11,
        "dynamic_inputs": ["prcp_e5l", "t_max_e5l", "t_min_e5l"],
        "config_sha256": "1b899b1807ddfebad35fbef0cf6da5bcd13deb0db294ff61e971ca2a0e561564",
        "checkpoint_sha256": "96d7fd8c87d8ec911f9272a6bb7fe6bac91840bfee49a5276748c703e054e39d",
        "scaler_sha256": "f215a2b5149b1e9aa25c0acef56aeac1cf84d66afc4a27d3b1b571e6ebdf5760",
    },
    "gpcp": {
        "label": "GPCP",
        "run_dir": DATA
        / "lstm_configs"
        / "model_runs"
        / "FULL_cudalstm_q_mm_day_256_365_gpcp_1201_212718",
        "epoch": 13,
        "dynamic_inputs": ["prcp_gpcp", "t_max_e5l", "t_min_e5l"],
        "config_sha256": "1934aa9e69907829c076057f8730dcde312058586cbd4b9a283f3464a17b3ccd",
        "checkpoint_sha256": "b21dadc3ba34de4d8e5d27b3f48f5f21a5b74ce84e60263709ee9e0a7a1881bc",
        "scaler_sha256": "a8b6852c5036abadff71436cb81ed1667bcd6fffcd319c6db836539d3d933754",
    },
    "mswep": {
        "label": "MSWEP",
        "run_dir": DATA
        / "lstm_configs"
        / "model_runs"
        / "FULL_cudalstm_q_mm_day_256_365_mswep_1301_013021",
        "epoch": 13,
        "dynamic_inputs": ["prcp_mswep", "t_max_e5l", "t_min_e5l"],
        "config_sha256": "91eb6b791600f0257e3ef5c7ed2489dd710465e1ffda28ddf0107d89d35464cd",
        "checkpoint_sha256": "8dacbcd538b955966cea452fd3d387c7f1f67dccdb1483294c623f7f024755e3",
        "scaler_sha256": "3e0813a3f2a95814f69f85474b9702d07f8471359d195e8f15f43b71b711206c",
    },
}
SOURCES_TO_RUN = tuple(MODEL_SPECS)
NOTEBOOK_STARTED_UTC = datetime.now(timezone.utc).isoformat()

print("Репозиторий:", ROOT)
print("Каталог результата:", RUN_ROOT)
print("Источники:", ", ".join(SOURCES_TO_RUN))


Репозиторий: /home/dmbrmv/Development/Dissertation
Каталог результата: /home/dmbrmv/Development/Dissertation/data/res_20260904/chapter_three/blind_transfer_recalc/2026-09-04_frozen_pre_update_v1
Источники: e5, e5l, gpcp, mswep


In [2]:
def sha256(path: Path) -> str:
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def atomic_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    if temporary.exists():
        raise FileExistsError(f"Остался временный файл: {temporary}")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def atomic_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    if temporary.exists():
        raise FileExistsError(f"Остался временный файл: {temporary}")
    frame.to_csv(temporary, index=False)
    temporary.replace(path)


def best_epoch(validation_dir: Path) -> tuple[int, float]:
    medians = {}
    for metrics_path in sorted(validation_dir.glob("model_epoch*/validation_metrics.csv")):
        epoch = int(metrics_path.parent.name.removeprefix("model_epoch"))
        medians[epoch] = float(pd.read_csv(metrics_path)["NSE"].median())
    if not medians:
        raise FileNotFoundError(f"Не найдены validation_metrics.csv: {validation_dir}")
    epoch = max(medians, key=medians.get)
    return epoch, medians[epoch]


def copy_frozen(source: Path, target: Path) -> str:
    expected = sha256(source)
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        if sha256(target) != expected:
            raise RuntimeError(f"Существующая копия отличается от источника: {target}")
    else:
        shutil.copy2(source, target)
    return expected


def verify_prediction_manifest(source: str) -> dict:
    source_dir = PREDICTIONS_DIR / source
    manifest_path = source_dir / "manifest.json"
    if not manifest_path.exists():
        raise FileNotFoundError(manifest_path)
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest["source"] != source or manifest["gauge_count"] != 409:
        raise RuntimeError(f"Неверный манифест {manifest_path}")
    for relative_path, expected_hash in manifest["files"].items():
        artifact = source_dir / relative_path
        if not artifact.exists() or sha256(artifact) != expected_hash:
            raise RuntimeError(f"Нарушен хэш прогноза: {artifact}")
    return manifest



In [5]:
# Эта ячейка намеренно прекращает запуск на CPU или с другой версией NeuralHydrology.
from neuralhydrology.evaluation import get_tester
from neuralhydrology.utils.config import Config
import torch

assert torch.cuda.is_available(), "Нужен CUDA-GPU; на Mac эту ячейку не запускать."
assert importlib.metadata.version("neuralhydrology") == "1.12.0", (
    "Нужна neuralhydrology==1.12.0 — версия, которой созданы checkpoints."
)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

for required in [
    OLD_TS,
    ATTRIBUTES_SOURCE,
    BLIND_SOURCE,
    INVENTORY_SOURCE,
    OBSERVATIONS_DIR,
]:
    assert required.exists(), required

blind_gauges = [line.strip() for line in BLIND_SOURCE.read_text().splitlines() if line.strip()]
inventory = pd.read_csv(INVENTORY_SOURCE, dtype={"gauge_id": str})
honest_gauges = inventory.loc[
    inventory["blind_role"].eq("honest_unseen"), "gauge_id"
].tolist()

assert len(blind_gauges) == len(set(blind_gauges)) == 409
assert set(inventory["gauge_id"]) == set(blind_gauges)
assert len(honest_gauges) == 395
# assert inventory["blind_role"].eq("overlap_primary_996").sum() == 14
assert all((OLD_TS / f"{gauge}.nc").exists() for gauge in blind_gauges)
assert all((OBSERVATIONS_DIR / f"{gauge}.nc").exists() for gauge in blind_gauges)

attributes = pd.read_csv(ATTRIBUTES_SOURCE, dtype={"gauge_id": str})
assert set(blind_gauges).issubset(set(attributes["gauge_id"]))

model_rows = []
for source, spec in MODEL_SPECS.items():
    config_path = spec["run_dir"] / "config.yml"
    checkpoint_path = spec["run_dir"] / f"model_epoch{spec['epoch']:03d}.pt"
    scaler_path = spec["run_dir"] / "train_data" / "train_data_scaler.yml"

    assert sha256(config_path) == spec["config_sha256"], config_path
    assert sha256(checkpoint_path) == spec["checkpoint_sha256"], checkpoint_path
    assert sha256(scaler_path) == spec["scaler_sha256"], scaler_path

    epoch, validation_median = best_epoch(spec["run_dir"] / "validation")
    assert epoch == spec["epoch"], (source, epoch, spec["epoch"])

    config = Config(config_path)
    assert list(config.dynamic_inputs) == spec["dynamic_inputs"], source
    assert config.test_start_date.strftime("%Y-%m-%d") == "2017-01-01"
    assert config.test_end_date.strftime("%Y-%m-%d") == "2018-12-31"

    model_rows.append(
        {
            "source": source,
            "epoch": epoch,
            "validation_median_NSE": validation_median,
            "dynamic_inputs": ", ".join(spec["dynamic_inputs"]),
        }
    )

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__, "| CUDA:", torch.version.cuda)
display(pd.DataFrame(model_rows))


GPU: NVIDIA GeForce RTX 4080 SUPER
PyTorch: 2.5.1+cu124 | CUDA: 12.4


,source,epoch,validation_median_NSE,dynamic_inputs
0,e5,12,0.629910,"prcp_e5, t_max_e5l, t_min_e5l"
1,e5l,11,0.642600,"prcp_e5l, t_max_e5l, t_min_e5l"
2,gpcp,13,0.588943,"prcp_gpcp, t_max_e5l, t_min_e5l"
3,mswep,13,0.666017,"prcp_mswep, t_max_e5l, t_min_e5l"


In [6]:
# Формируется отдельный data-root NeuralHydrology. Исходные NetCDF не изменяются.
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CONTRACTS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

frozen_blind = CONTRACTS_DIR / "blind_gauges.txt"
frozen_inventory = CONTRACTS_DIR / "blind_inventory_audit.csv"
copy_frozen(BLIND_SOURCE, frozen_blind)
copy_frozen(INVENTORY_SOURCE, frozen_inventory)

if INPUT_MANIFEST.exists():
    input_manifest = json.loads(INPUT_MANIFEST.read_text(encoding="utf-8"))
    assert input_manifest["contract_version"] == "frozen_pre_update_inputs_v1"
    assert input_manifest["gauge_count"] == 409
    for gauge, metadata in input_manifest["files"].items():
        staged = FROZEN_TS / f"{gauge}.nc"
        assert staged.exists() and sha256(staged) == metadata["staged_sha256"], staged
    copied_attributes = FROZEN_ATTRIBUTES / ATTRIBUTES_SOURCE.name
    assert sha256(copied_attributes) == input_manifest["attributes_sha256"]
    print("Замороженный data-root уже существует и прошёл проверку хэшей.")
else:
    FROZEN_TS.mkdir(parents=True, exist_ok=True)
    FROZEN_ATTRIBUTES.mkdir(parents=True, exist_ok=True)
    existing = list(FROZEN_TS.iterdir())
    if existing:
        raise RuntimeError(
            f"Найден незавершённый data-root ({len(existing)} файлов). "
            "Не удаляйте его вслепую: смените RUN_TAG или проверьте вручную."
        )

    attributes_hash = copy_frozen(
        ATTRIBUTES_SOURCE,
        FROZEN_ATTRIBUTES / ATTRIBUTES_SOURCE.name,
    )
    input_files = {}

    for number, gauge in enumerate(blind_gauges, start=1):
        source_path = OLD_TS / f"{gauge}.nc"
        target_path = FROZEN_TS / f"{gauge}.nc"
        temporary_path = target_path.with_name(target_path.name + ".tmp")

        with xr.open_dataset(source_path) as opened:
            dataset = opened.load()

        if "date" not in dataset.coords:
            if "index" not in dataset.coords:
                raise ValueError(f"Нет координаты index/date: {source_path}")
            dataset = dataset.rename({"index": "date"})

        if "gauge_id" in dataset.dims:
            if dataset.sizes["gauge_id"] != 1:
                raise ValueError(f"Неожиданный размер gauge_id: {source_path}")
            dataset = dataset.squeeze("gauge_id", drop=True)
        elif "gauge_id" in dataset.variables:
            dataset = dataset.drop_vars("gauge_id")

        missing_variables = sorted(set(REQUIRED_VARIABLES) - set(dataset.data_vars))
        if missing_variables:
            raise ValueError(f"{source_path}: отсутствуют {missing_variables}")

        staged = dataset[REQUIRED_VARIABLES].sel(date=INPUT_DATES).sortby("date")
        staged_dates = pd.DatetimeIndex(staged["date"].values)
        assert staged_dates.equals(INPUT_DATES), source_path
        assert not staged_dates.has_duplicates
        assert all(staged[variable].dims == ("date",) for variable in REQUIRED_VARIABLES)

        if temporary_path.exists():
            raise FileExistsError(temporary_path)
        staged.to_netcdf(temporary_path, engine="netcdf4")
        temporary_path.replace(target_path)

        with xr.open_dataset(target_path) as checked:
            assert set(checked.data_vars) == set(REQUIRED_VARIABLES)
            assert pd.DatetimeIndex(checked["date"].values).equals(INPUT_DATES)

        input_files[gauge] = {
            "source_sha256": sha256(source_path),
            "staged_sha256": sha256(target_path),
        }
        if number % 50 == 0 or number == len(blind_gauges):
            print(f"Подготовлено {number}/{len(blind_gauges)} NetCDF")

    input_manifest = {
        "contract_version": "frozen_pre_update_inputs_v1",
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "source_root": str(OLD_TS.resolve()),
        "staged_root": str(FROZEN_DATA.resolve()),
        "gauge_count": len(blind_gauges),
        "input_period": {
            "start": str(INPUT_DATES[0].date()),
            "end": str(INPUT_DATES[-1].date()),
            "days": len(INPUT_DATES),
        },
        "variables": REQUIRED_VARIABLES,
        "attributes_sha256": attributes_hash,
        "blind_gauges_sha256": sha256(frozen_blind),
        "inventory_sha256": sha256(frozen_inventory),
        "normalization": "coordinate index renamed to date; values and NaN masks preserved",
        "files": input_files,
    }
    atomic_json(INPUT_MANIFEST, input_manifest)
    print("Создан манифест входов:", INPUT_MANIFEST)

print("SHA-256 манифеста входов:", sha256(INPUT_MANIFEST))


Подготовлено 50/409 NetCDF
Подготовлено 100/409 NetCDF
Подготовлено 150/409 NetCDF
Подготовлено 200/409 NetCDF
Подготовлено 250/409 NetCDF
Подготовлено 300/409 NetCDF
Подготовлено 350/409 NetCDF
Подготовлено 400/409 NetCDF
Подготовлено 409/409 NetCDF
Создан манифест входов: /home/dmbrmv/Development/Dissertation/data/res_20260904/chapter_three/blind_transfer_recalc/2026-09-04_frozen_pre_update_v1/neuralhydrology_data/manifest.json
SHA-256 манифеста входов: 3c4585f9e8661255ef4b8c7ea0410cc413a28887d37d701395b6f50c072b179a


In [7]:
def run_source(source: str) -> dict:
    spec = MODEL_SPECS[source]
    final_dir = PREDICTIONS_DIR / source
    partial_dir = PREDICTIONS_DIR / f".{source}.partial"

    if final_dir.exists():
        manifest = verify_prediction_manifest(source)
        print(f"{source}: готовый результат подтверждён, повторный запуск пропущен.")
        return manifest
    if partial_dir.exists():
        raise RuntimeError(
            f"Обнаружен незавершённый расчёт {partial_dir}. "
            "Проверьте его вручную; notebook ничего не удаляет."
        )

    partial_dir.mkdir(parents=True)
    config_path = spec["run_dir"] / "config.yml"
    existing_test_result = (
        spec["run_dir"]
        / "test"
        / f"model_epoch{spec['epoch']:03d}"
        / "test_results.p"
    )
    existing_test_hash = sha256(existing_test_result) if existing_test_result.exists() else None

    config = Config(config_path)
    config.update_config(
        {
            "data_dir": str(FROZEN_DATA),
            "test_basin_file": str(frozen_blind),
            "test_start_date": "01/01/2017",
            "test_end_date": "31/12/2018",
            "device": "cuda:0",
        }
    )

    started_utc = datetime.now(timezone.utc).isoformat()
    tester = get_tester(
        cfg=config,
        run_dir=spec["run_dir"],
        period="test",
        init_model=True,
    )
    # save_results=False не позволяет NeuralHydrology затереть старый test_results.p.
    raw_results = tester.evaluate(epoch=spec["epoch"], save_results=False)
    results = {str(gauge): value for gauge, value in raw_results.items()}
    assert set(results) == set(blind_gauges), (
        source,
        len(results),
        sorted(set(blind_gauges) - set(results))[:10],
    )

    files = {}
    for number, gauge in enumerate(blind_gauges, start=1):
        result_dataset = results[gauge]["1D"]["xr"]
        daily = result_dataset[
            ["q_mm_day_obs", "q_mm_day_sim"]
        ].to_dataframe().reset_index()
        daily["date"] = pd.to_datetime(daily["date"])
        daily = (
            daily[["date", "q_mm_day_obs", "q_mm_day_sim"]]
            .rename(columns={"q_mm_day_obs": "q_obs", "q_mm_day_sim": "q_sim"})
            .sort_values("date")
            .reset_index(drop=True)
        )

        assert len(daily) == len(EVALUATION_DATES), (source, gauge, len(daily))
        assert not daily["date"].duplicated().any(), (source, gauge)
        assert pd.DatetimeIndex(daily["date"]).equals(EVALUATION_DATES), (source, gauge)

        relative_path = Path(gauge) / f"{gauge}_{source}_predictions.csv"
        output_path = partial_dir / relative_path
        output_path.parent.mkdir(parents=True)
        daily.to_csv(output_path, index=False)
        files[str(relative_path)] = sha256(output_path)

        if number % 100 == 0 or number == len(blind_gauges):
            print(f"{source}: записано {number}/{len(blind_gauges)} CSV")

    if existing_test_result.exists():
        assert sha256(existing_test_result) == existing_test_hash, (
            f"NeuralHydrology изменил старый test_results.p: {existing_test_result}"
        )

    manifest = {
        "contract_version": "blind_transfer_frozen_inference_v1",
        "source": source,
        "source_label": spec["label"],
        "started_utc": started_utc,
        "finished_utc": datetime.now(timezone.utc).isoformat(),
        "gauge_count": len(results),
        "period": {
            "start": str(EVALUATION_DATES[0].date()),
            "end": str(EVALUATION_DATES[-1].date()),
            "days": len(EVALUATION_DATES),
        },
        "epoch": spec["epoch"],
        "dynamic_inputs": spec["dynamic_inputs"],
        "config_path": str(config_path.resolve()),
        "config_sha256": spec["config_sha256"],
        "checkpoint_sha256": spec["checkpoint_sha256"],
        "scaler_sha256": spec["scaler_sha256"],
        "input_manifest_sha256": sha256(INPUT_MANIFEST),
        "neuralhydrology_save_results": False,
        "files": files,
    }
    atomic_json(partial_dir / "manifest.json", manifest)
    partial_dir.rename(final_dir)
    verify_prediction_manifest(source)

    del raw_results, results, tester, config
    gc.collect()
    torch.cuda.empty_cache()
    print(f"{source}: завершено -> {final_dir}")
    return manifest


prediction_manifests = {}
for source in SOURCES_TO_RUN:
    prediction_manifests[source] = run_source(source)

display(
    pd.DataFrame(
        [
            {
                "source": source,
                "epoch": manifest["epoch"],
                "gauges": manifest["gauge_count"],
                "finished_utc": manifest["finished_utc"],
            }
            for source, manifest in prediction_manifests.items()
        ]
    )
)


/home/dmbrmv/Development/Dissertation/.pixi/envs/default/lib/python3.12/site-packages/neuralhydrology/evaluation/tester.py:143: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
 

# Evaluation:  97%|█████████▋| 395/409 [00:34<00:01, 12.18it/s]

The following basins had not enough valid target values to calculate a standard deviation: 9319. NSE loss values for this basin will be NaN.


# Evaluation: 100%|██████████| 409/409 [00:35<00:00, 11.47it/s]
e5: записано 100/409 CSV
e5: записано 200/409 CSV
e5: записано 300/409 CSV


/home/dmbrmv/Development/Dissertation/.pixi/envs/default/lib/python3.12/site-packages/neuralhydrology/evaluation/tester.py:143: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
 

...

/home/dmbrmv/Development/Dissertation/.pixi/envs/default/lib/python3.12/site-packages/neuralhydrology/evaluation/tester.py:143: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
 

gpcp: записано 400/409 CSV
gpcp: записано 409/409 CSV
gpcp: завершено -> /home/dmbrmv/Development/Dissertation/data/res_20260904/chapter_three/blind_transfer_recalc/2026-09-04_frozen_pre_update_v1/predictions/gpcp
# Evaluation:  97%|█████████▋| 396/409 [00:34<00:01, 11.03it/s]

The following basins had not enough valid target values to calculate a standard deviation: 9319. NSE loss values for this basin will be NaN.


# Evaluation: 100%|██████████| 409/409 [00:35<00:00, 11.60it/s]
mswep: записано 100/409 CSV
mswep: записано 200/409 CSV
mswep: записано 300/409 CSV
mswep: записано 400/409 CSV
mswep: записано 409/409 CSV
mswep: завершено -> /home/dmbrmv/Development/Dissertation/data/res_20260904/chapter_three/blind_transfer_recalc/2026-09-04_frozen_pre_update_v1/predictions/mswep


,source,epoch,gauges,finished_utc
0,e5,12,409,2026-09-03T01:35:19.649931+00:00
1,e5l,11,409,2026-09-03T01:35:55.608020+00:00
2,gpcp,13,409,2026-09-03T01:36:31.921330+00:00
3,mswep,13,409,2026-09-03T01:37:08.178302+00:00


In [8]:
from src.timeseries_stats.metrics import evaluate_model

frozen_inventory_frame = pd.read_csv(frozen_inventory, dtype={"gauge_id": str})
honest_gauges = frozen_inventory_frame.loc[
    frozen_inventory_frame["blind_role"].eq("honest_unseen"), "gauge_id"
].tolist()
assert len(honest_gauges) == 395

metric_names = ["NSE", "KGE", "PBIAS", "RMSE", "MAE", "logNSE", "R2", "r", "PFE"]
metric_rows = []

for source in SOURCES_TO_RUN:
    verify_prediction_manifest(source)
    for gauge in honest_gauges:
        prediction_path = (
            PREDICTIONS_DIR
            / source
            / gauge
            / f"{gauge}_{source}_predictions.csv"
        )
        daily = pd.read_csv(prediction_path, parse_dates=["date"]).sort_values("date")
        assert len(daily) == len(EVALUATION_DATES), prediction_path
        assert pd.DatetimeIndex(daily["date"]).equals(EVALUATION_DATES), prediction_path

        with xr.open_dataset(OBSERVATIONS_DIR / f"{gauge}.nc") as observation_dataset:
            observation = (
                observation_dataset["q_mm_day"]
                .sel(date=EVALUATION_DATES)
                .to_numpy()
                .astype(float)
            )

        saved_observation = daily["q_obs"].to_numpy(dtype=float)
        simulation = daily["q_sim"].to_numpy(dtype=float)
        finite_difference = np.isfinite(observation) & np.isfinite(saved_observation)
        max_observation_difference = (
            float(np.max(np.abs(observation[finite_difference] - saved_observation[finite_difference])))
            if finite_difference.any()
            else np.nan
        )
        nan_masks_match = np.array_equal(
            np.isnan(observation),
            np.isnan(saved_observation),
        )
        saved_observation_matches = bool(
            nan_masks_match
            and np.allclose(
                observation,
                saved_observation,
                rtol=0,
                atol=2e-5,
                equal_nan=True,
            )
        )

        n_non_finite_observation = int((~np.isfinite(observation)).sum())
        n_non_finite_simulation = int((~np.isfinite(simulation)).sum())
        n_negative_observation = int((observation < 0).sum())
        n_negative_simulation = int((simulation < 0).sum())
        n_pairs = int((np.isfinite(observation) & np.isfinite(simulation)).sum())

        if not saved_observation_matches:
            status = "saved_observation_mismatch"
        elif n_negative_observation:
            status = "negative_observed_flow"
        elif n_non_finite_observation and n_non_finite_simulation:
            status = "non_finite_observed_and_simulated"
        elif n_non_finite_observation:
            status = "non_finite_observed"
        elif n_non_finite_simulation:
            status = "non_finite_simulated"
        else:
            status = "ok"

        metrics = (
            evaluate_model(observed=observation, simulated=simulation)
            if status == "ok"
            else dict.fromkeys(metric_names, np.nan)
        )
        metric_rows.append(
            {
                "gauge_id": gauge,
                "model": "lstm",
                "meteo": source,
                "period": "honest_blind_frozen_pre_update",
                "period_start": str(EVALUATION_DATES[0].date()),
                "period_end": str(EVALUATION_DATES[-1].date()),
                "expected_days": len(EVALUATION_DATES),
                "source_file": str(prediction_path.resolve()),
                "source_sha256": sha256(prediction_path),
                "n_rows": len(daily),
                "n_pairs": n_pairs,
                "n_non_finite_obs": n_non_finite_observation,
                "n_non_finite_sim": n_non_finite_simulation,
                "n_negative_obs": n_negative_observation,
                "n_negative_sim": n_negative_simulation,
                "saved_q_obs_max_abs_diff": max_observation_difference,
                "saved_q_obs_nan_mask_matches": nan_masks_match,
                "saved_q_obs_within_tolerance": saved_observation_matches,
                **metrics,
                "status": status,
            }
        )

metrics = pd.DataFrame(metric_rows)
metrics_path = TABLES_DIR / "honest_blind_lstm_metrics_frozen_pre_update.csv"
atomic_csv(metrics, metrics_path)

valid = metrics.loc[metrics["status"].eq("ok")]
summary = (
    valid.groupby("meteo", sort=False)
    .agg(
        n_valid=("gauge_id", "size"),
        NSE_median=("NSE", "median"),
        KGE_median=("KGE", "median"),
        PBIAS_median=("PBIAS", "median"),
        RMSE_median=("RMSE", "median"),
        r_median=("r", "median"),
    )
    .reindex(SOURCES_TO_RUN)
    .reset_index()
)
summary["source_label"] = summary["meteo"].map(
    {source: spec["label"] for source, spec in MODEL_SPECS.items()}
)

if PREVIOUS_METRICS.exists():
    previous = pd.read_csv(PREVIOUS_METRICS, dtype={"gauge_id": str})
    previous_valid = previous.loc[previous["status"].eq("ok")]
    previous_summary = (
        previous_valid.groupby("meteo")
        .agg(
            previous_n_valid=("gauge_id", "size"),
            previous_NSE_median=("NSE", "median"),
            previous_PBIAS_median=("PBIAS", "median"),
        )
        .reset_index()
    )
    summary = summary.merge(previous_summary, on="meteo", how="left")
    summary["NSE_delta"] = summary["NSE_median"] - summary["previous_NSE_median"]
    summary["PBIAS_delta"] = summary["PBIAS_median"] - summary["previous_PBIAS_median"]

summary_path = TABLES_DIR / "blind_transfer_summary_frozen_pre_update.csv"
atomic_csv(summary, summary_path)

notebook_path = ROOT / "notebooks" / "c3_RecalculateBlindTransfer_FrozenInputs.ipynb"
completed = {
    "contract_version": "chapter3_blind_transfer_frozen_pre_update_v1",
    "started_utc": NOTEBOOK_STARTED_UTC,
    "finished_utc": datetime.now(timezone.utc).isoformat(),
    "run_tag": RUN_TAG,
    "runtime": {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "xarray": xr.__version__,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "neuralhydrology": importlib.metadata.version("neuralhydrology"),
    },
    "notebook_sha256": sha256(notebook_path) if notebook_path.exists() else None,
    "input_manifest": str(INPUT_MANIFEST.resolve()),
    "input_manifest_sha256": sha256(INPUT_MANIFEST),
    "prediction_manifests": {
        source: {
            "path": str((PREDICTIONS_DIR / source / "manifest.json").resolve()),
            "sha256": sha256(PREDICTIONS_DIR / source / "manifest.json"),
        }
        for source in SOURCES_TO_RUN
    },
    "artifacts": {
        str(metrics_path.relative_to(RUN_ROOT)): sha256(metrics_path),
        str(summary_path.relative_to(RUN_ROOT)): sha256(summary_path),
    },
    "strict_valid_rows_by_meteo": (
        valid.groupby("meteo").size().reindex(SOURCES_TO_RUN).fillna(0).astype(int).to_dict()
    ),
}
atomic_json(COMPLETED_MANIFEST, completed)

display(summary.round(3))
print("Метрики:", metrics_path)
print("Сводка:", summary_path)
print("Финальный манифест:", COMPLETED_MANIFEST)


,meteo,n_valid,NSE_median,KGE_median,PBIAS_median,RMSE_median,r_median,source_label,previous_n_valid,previous_NSE_median,previous_PBIAS_median,NSE_delta,PBIAS_delta
0,e5,303,0.568,0.556,-7.022,0.664,0.855,ERA5,303,0.559,-6.993,0.009,-0.029
1,e5l,305,0.547,0.526,-2.648,0.642,0.837,ERA5-Land,305,-0.119,72.500,0.667,-75.147
2,gpcp,305,0.520,0.528,0.872,0.688,0.836,GPCP,305,0.519,1.029,0.001,-0.157
3,mswep,305,0.582,0.545,-3.950,0.615,0.857,MSWEP,305,0.582,-3.902,-0.000,-0.048


Метрики: /home/dmbrmv/Development/Dissertation/data/res_20260904/chapter_three/blind_transfer_recalc/2026-09-04_frozen_pre_update_v1/tables/honest_blind_lstm_metrics_frozen_pre_update.csv
Сводка: /home/dmbrmv/Development/Dissertation/data/res_20260904/chapter_three/blind_transfer_recalc/2026-09-04_frozen_pre_update_v1/tables/blind_transfer_summary_frozen_pre_update.csv
Финальный манифест: /home/dmbrmv/Development/Dissertation/data/res_20260904/chapter_three/blind_transfer_recalc/2026-09-04_frozen_pre_update_v1/COMPLETED.json


## После выполнения

1. Сохраните вывод итоговой таблицы и файл `COMPLETED.json`.
2. Не копируйте новые значения в главу автоматически: сначала сравните четыре варианта и строгие размеры выборок.
3. Для обновления главы используйте таблицу `honest_blind_lstm_metrics_frozen_pre_update.csv`; старые прогнозы остаются нетронутыми.
4. Если расчёт прерван, notebook не удаляет частичный каталог `.<source>.partial`. Проверьте его и либо задайте новый `RUN_TAG`, либо удалите только подтверждённый частичный каталог вручную.